# Interpretable Concept Steering via Sparse Autoencoders

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/27Kushal/sae-concept-steering/blob/main/notebooks/colab_sae_pipeline.ipynb)

This notebook runs the **full-scale** pipeline for training a Sparse Autoencoder (SAE) on the residual stream of GPT-2 small (Layer 6) using **tens of millions of tokens** on a Google Colab T4 GPU (or A100/V100).

### Pipeline Overview:
1. **Activation Collection**: Sharded extraction of 10M tokens from GPT-2 residual stream (`blocks.6.hook_resid_post`).
2. **SAE Training**: Train overcomplete SAE ($8\times$ expansion: $768 \to 6144 \to 768$) with strictly unit-norm constrained decoder columns, tracking L0, FVE, and dead features.
3. **Feature Interpretation**: Systematic (non-cherry-picked) top-$k$ snippet extraction and hypothesis validation.
4. **Activation Steering & Quantitative Evaluation**: Hooked generation sweeping steering strength $\alpha$ and benchmarking efficacy and collateral damage against Difference-of-Means baseline.
5. **Checkpoint Download**: Package trained checkpoint to run lightweight inference locally.

## 1. Hardware & Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install dependencies
!pip install -q transformers>=4.38.0 transformer_lens>=1.14.0 datasets>=2.16.0 einops jaxtyping pandas scipy scikit-learn tqdm matplotlib

## 2. Clone Repository or Mount Code

In [ ]:
# Clone repository if running in fresh Colab session
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/27Kushal/sae-concept-steering.git
    %cd sae-concept-steering
sys.path.insert(0, os.path.abspath("."))


## 3. Configuration Setup (Full Scale)

In [ ]:
from src.config import ProjectConfig, get_device

# Load full-scale preset (10,000,000 tokens, 10,000 steps, batch size 4096)
config = ProjectConfig.create(scale="full")
device = get_device()
print(f"Target scale: {config.scale}")
print(f"Target tokens: {config.data.total_tokens:,}")
print(f"SAE dimension: {config.sae.d_sae} (8x expansion)")
print(f"Resolved device: {device}")

## 4. Activation Collection & Sharded Disk Caching

In [ ]:
from src.data import collect_and_shard_activations

# Collect activations from GPT-2 small Layer 6 to disk shards
shard_paths = collect_and_shard_activations(
    data_config=config.data,
    model_config=config.model,
    device=device,
)
print(f"Successfully generated {len(shard_paths)} shards in {config.data.cache_dir}")

## 5. Train Sparse Autoencoder (SAE)

In [ ]:
from src.sae import SparseAutoencoder
from src.data import ShardedActivationBuffer
from src.train import train_sae

sae = SparseAutoencoder(config.sae)
train_buffer = ShardedActivationBuffer(
    shard_paths=shard_paths,
    batch_size=config.train.batch_size,
    device=device,
    shuffle=True,
)

summary = train_sae(
    sae=sae,
    train_buffer=train_buffer,
    val_shard_path=shard_paths[-1],
    train_config=config.train,
    device=device,
)

## 6. Plot Training Curves (Loss, L0, FVE)

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in summary["history"]]
l2_losses = [h["l2_loss"] for h in summary["history"]]
l0_vals = [h["l0"] for h in summary["history"]]
fve_vals = [h["fve"] for h in summary["history"]]

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
axs[0].plot(steps, l2_losses, color='tab:blue')
axs[0].set_title("L2 Reconstruction Loss")
axs[0].set_xlabel("Step")

axs[1].plot(steps, l0_vals, color='tab:orange')
axs[1].set_title("L0 Sparsity (Active Latents / Token)")
axs[1].set_xlabel("Step")

axs[2].plot(steps, fve_vals, color='tab:green')
axs[2].set_title("Fraction of Variance Explained (FVE)")
axs[2].set_xlabel("Step")

plt.tight_layout()
plt.show()

## 7. Systematic Feature Interpretation (100 Sampled Latents)

In [ ]:
from src.model import load_hooked_transformer
from src.interpret import run_systematic_feature_interpretation

model, _ = load_hooked_transformer(config.model, device=device)
val_acts = torch.load(shard_paths[0], map_location='cpu')[:8192].to(device)

# Sample evaluation texts
eval_texts = [
    "def compute_loss(pred, target): return torch.nn.functional.mse_loss(pred, target)",
    "The deep learning research community has embraced sparse autoencoders for interpretability.",
    "import numpy as np; data = np.random.randn(100, 100)",
    "The architectural heritage of the ancient city continues to inspire architects worldwide."
] * 10

report = run_systematic_feature_interpretation(
    model=model,
    sae=sae,
    val_acts=val_acts,
    sample_texts=eval_texts,
    output_report_path="results/colab_feature_interpretability_report.json",
    n_top=50,
    n_random=50,
)

## 8. Activation Steering & Quantitative Evaluation vs Baseline

In [ ]:
from src.steering import compute_difference_of_means_vector, run_steering_sweep
from src.evaluate import run_full_comparative_evaluation

# Define target concept and contrastive prompts directly
concept_name = "python_code"
concept_data = {
    "feature_idx": 42,
    "test_prompts": [
        "Write a function that",
        "Here is the implementation:",
        "To solve this problem in software,"
    ],
    "positive_prompts": [
        "def calculate_mean(numbers):\n    return sum(numbers) / len(numbers)",
        "import os\nimport sys\nfrom pathlib import Path",
        "class NeuralNet(torch.nn.Module):\n    def __init__(self):",
        "for i in range(len(items)):\n    if items[i] == target: return i"
    ],
    "negative_prompts": [
        "The morning sun cast a gentle golden glow over the tranquil valley.",
        "Historical trade routes connected civilizations across ancient continents.",
        "Fresh ingredients and careful seasoning are the secrets to great cuisine.",
        "The orchestra played a moving symphony that echoed through the grand concert hall."
    ]
}
target_feature_idx = concept_data["feature_idx"]

# SAE feature direction
sae_vector = sae.get_feature_direction(target_feature_idx).to(device)

# Difference of means baseline
diff_vector = compute_difference_of_means_vector(
    model=model,
    positive_prompts=concept_data["positive_prompts"],
    negative_prompts=concept_data["negative_prompts"],
    device=device,
).to(device)

alphas = [-8.0, -4.0, 0.0, 4.0, 8.0]
sae_samples = run_steering_sweep(model, concept_data["test_prompts"], sae_vector, alphas, device=device)
diff_samples = run_steering_sweep(model, concept_data["test_prompts"], diff_vector, alphas, device=device)

eval_df = run_full_comparative_evaluation(model, concept_name, sae_samples, diff_samples, device=device)
summary_df = eval_df.groupby(["method", "alpha"]).agg({
    "judge_score": "mean",
    "perplexity": "mean",
    "perplexity_delta": "mean",
}).reset_index()

print(summary_df.to_markdown(index=False))


## 9. Package & Download Checkpoint

In [ ]:
!zip -r sae_colab_checkpoint.zip checkpoints/full/final/
from google.colab import files
files.download("sae_colab_checkpoint.zip")